In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from typing import Final
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
window_size: Final[int] = 1
horizon: Final[int] = 1

EPOCHS: Final[int] = 50
BATCH_SIZE: Final[int] = 32

In [8]:
def evaluate_model(model, x_test, y_test):
    predictions = model.predict(x_test)
    print(f"MSE: {mean_squared_error(y_test, predictions)}")
    print(f"MAE: {mean_absolute_error(y_test, predictions)}")
    print(f"R2: {r2_score(y_test, predictions)}")

In [9]:
df = pd.read_csv("DailyDelhiClimateTrain.csv")

df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Shape: {df.shape}")

display(df)

Shape: (1462, 4)


,meantemp,humidity,wind_speed,meanpressure
date,,,,
2013-01-01,10.000000,84.500000,0.000000,1015.666667
2013-01-02,7.400000,92.000000,2.980000,1017.800000
2013-01-03,7.166667,87.000000,4.633333,1018.666667
2013-01-04,8.666667,71.333333,1.233333,1017.166667
2013-01-05,6.000000,86.833333,3.700000,1016.500000
...,...,...,...,...
2016-12-28,17.217391,68.043478,3.547826,1015.565217
2016-12-29,15.238095,87.857143,6.000000,1016.904762
2016-12-30,14.095238,89.666667,6.266667,1017.904762


In [10]:
series = df["humidity"].values.reshape(-1, 1)
scaler = MinMaxScaler()
series_data = scaler.fit_transform(series)

In [11]:
def make_dataset(data, window_size, horizon):
    x, y = [], []
    for i in range(len(data) - window_size - horizon):
        x.append(data[i:i+window_size])
        y.append(data[i+window_size + horizon])
    return np.array(x), np.array(y)

x, y = make_dataset(series_data, window_size, horizon)
x = x[..., None]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

In [12]:
rnn_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.SimpleRNN(64),
    layers.Dense(horizon)
])
rnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = rnn_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0965 - mae: 0.2454 - val_loss: 0.0162 - val_mae: 0.0998
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0181 - mae: 0.1079 - val_loss: 0.0161 - val_mae: 0.0985
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0163 - mae: 0.1016 - val_loss: 0.0156 - val_mae: 0.0965
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0157 - mae: 0.0996 - val_loss: 0.0156 - val_mae: 0.0961
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0153 - mae: 0.0978 - val_loss: 0.0150 - val_mae: 0.0943
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0148 - mae: 0.0961 - val_loss: 0.0151 - val_mae: 0.0939
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0145 - mae: 0.0947 - val_loss: 0.0151 - val_mae: 0.0936
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0144 - mae: 0.0939 - val_loss: 0.0152 - val_mae: 0.0933
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0143 - mae:

In [13]:
evaluate_model(rnn_model, x_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
MSE: 0.015175556100379446
MAE: 0.09316788171908107
R2: 0.5367807236107385


In [14]:
lstm_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.LSTM(64),
    layers.Dense(horizon)
])
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = lstm_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.2156 - mae: 0.4251 - val_loss: 0.1460 - val_mae: 0.3497
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0773 - mae: 0.2382 - val_loss: 0.0389 - val_mae: 0.1660
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0234 - mae: 0.1231 - val_loss: 0.0177 - val_mae: 0.1039
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0189 - mae: 0.1097 - val_loss: 0.0172 - val_mae: 0.1025
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0184 - mae: 0.1083 - val_loss: 0.0170 - val_mae: 0.1017
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0180 - mae: 0.1071 - val_loss: 0.0167 - val_mae: 0.1007
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0176 - mae: 0.1056 - val_loss: 0.0165 - val_mae: 0.0998
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0172 - mae: 0.1045 - val_loss: 0.0163 - val_mae: 0.0993
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0168 - mae:

In [15]:
evaluate_model(lstm_model, x_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
MSE: 0.015285566052190672
MAE: 0.09407286589175728
R2: 0.5334227754777972


In [16]:
gru_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.GRU(64),
    layers.Dense(horizon)
])
gru_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = gru_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.1377 - mae: 0.3196 - val_loss: 0.0357 - val_mae: 0.1579
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0217 - mae: 0.1173 - val_loss: 0.0182 - val_mae: 0.1060
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0201 - mae: 0.1133 - val_loss: 0.0180 - val_mae: 0.1051
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0190 - mae: 0.1101 - val_loss: 0.0175 - val_mae: 0.1032
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0182 - mae: 0.1073 - val_loss: 0.0167 - val_mae: 0.1006
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0174 - mae: 0.1049 - val_loss: 0.0163 - val_mae: 0.0991
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0166 - mae: 0.1026 - val_loss: 0.0161 - val_mae: 0.0981
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0160 - mae: 0.1006 - val_loss: 0.0159 - val_mae: 0.0971
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0155 - mae:

In [17]:
evaluate_model(gru_model, x_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
MSE: 0.01643936898369663
MAE: 0.09619022091604423
R2: 0.49820404902765836
